# 01 Deploy Infrastructure

This notebook deploys one resource group in UK South with:
- Classic Azure OpenAI account (kind OpenAI) with gpt-4o and gpt-5.1 deployments
- APIM gateway exposing POST /openai/chat/completions
- APIM operation policy forcing backend api-version=2024-10-21

Then it writes env/.env from deployment outputs and retrieved keys.

## Configuration

In [ ]:
import json
import os
import subprocess
from datetime import datetime, timezone
from pathlib import Path

DEMO_ID_BASE = 'evalgw02'
DEBUG_RUN = 1
DEMO_ID = DEMO_ID_BASE + (f'-{DEBUG_RUN}' if DEBUG_RUN else '')

LOCATION = 'uksouth'
RG_NAME = f'rg-{DEMO_ID}-uks'
BICEP_FILE = '../infra/main.bicep'
DEPLOYMENT_NAME = f'main-{DEMO_ID}'

MODEL_NAME = 'gpt-4o'
MODEL_VERSION = '2024-11-20'
PRIMARY_DEPLOYMENT_NAME = 'chat4o'
SECONDARY_MODEL_NAME = 'gpt-5.1'
SECONDARY_MODEL_VERSION = '2025-11-13'
SECONDARY_DEPLOYMENT_NAME = 'chat51'

print(f'Demo ID: {DEMO_ID}')
print(f'Resource group: {RG_NAME}')
print(f'Deployment name: {DEPLOYMENT_NAME}')
print(f'Location: {LOCATION}')
print(f'Primary model: {MODEL_NAME} ({MODEL_VERSION}) as {PRIMARY_DEPLOYMENT_NAME}')
print(f'Secondary model: {SECONDARY_MODEL_NAME} ({SECONDARY_MODEL_VERSION}) as {SECONDARY_DEPLOYMENT_NAME}')

## Create Resource Group

In [ ]:
result = subprocess.run(
    f'az group create --name {RG_NAME} --location {LOCATION} --tags SecurityControl=Ignore',
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    raise RuntimeError('Failed to create resource group:\n' + result.stderr)

print(f'Created or updated resource group: {RG_NAME}')
print('Applied tag: SecurityControl=Ignore')

## Deploy Bicep

In [ ]:
deploy_cmd = (
    f'az deployment group create '
    f'--resource-group {RG_NAME} '
    f'--name {DEPLOYMENT_NAME} '
    f'--template-file {BICEP_FILE} '
    f'--parameters demoId={DEMO_ID} location={LOCATION} '
    f'modelName={MODEL_NAME} modelVersion={MODEL_VERSION} deploymentName={PRIMARY_DEPLOYMENT_NAME} '
    f'secondaryModelName={SECONDARY_MODEL_NAME} secondaryModelVersion={SECONDARY_MODEL_VERSION} secondaryDeploymentName={SECONDARY_DEPLOYMENT_NAME} '
    f'--query properties.outputs -o json'
)

print('Deploying infrastructure...')
result = subprocess.run(deploy_cmd, shell=True, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Bicep deployment failed.')

if not result.stdout.strip():
    raise RuntimeError('Deployment returned no outputs.')

outputs = json.loads(result.stdout)
print('Deployment completed')
print(json.dumps(outputs, indent=2))

## Retrieve Keys

In [ ]:
if 'outputs' not in globals():
    raise RuntimeError('Deployment outputs not found. Run the Deploy Bicep cell first.')

deployed_rg = outputs.get('resourceGroupName', {}).get('value', '')
if deployed_rg != RG_NAME:
    raise RuntimeError(
        f'Deployment outputs are for {deployed_rg}, but current configuration expects {RG_NAME}. '
        'Re-run Configuration and Deploy Bicep cells so DEBUG_RUN follows through.'
    )

openai_name = outputs['openAiAccountName']['value']
apim_name = outputs['apimServiceName']['value']

openai_key_result = subprocess.run(
    f'az cognitiveservices account keys list --name {openai_name} --resource-group {RG_NAME} --query key1 -o tsv',
    shell=True,
    capture_output=True,
    text=True
)
if openai_key_result.returncode != 0 or not openai_key_result.stdout.strip():
    raise RuntimeError('Failed to retrieve Azure OpenAI key.')
openai_key = openai_key_result.stdout.strip()

subscription_result = subprocess.run(
    'az account show --query id -o tsv',
    shell=True,
    capture_output=True,
    text=True
)
if subscription_result.returncode != 0 or not subscription_result.stdout.strip():
    raise RuntimeError('Failed to resolve Azure subscription id from az account show.')
subscription_id = subscription_result.stdout.strip()

subs_url = (
    f'https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{RG_NAME}/'
    f'providers/Microsoft.ApiManagement/service/{apim_name}/subscriptions?api-version=2022-08-01'
)
subs_result = subprocess.run(
    f'az rest --method get --url "{subs_url}" -o json',
    shell=True,
    capture_output=True,
    text=True
)
if subs_result.returncode != 0 or not subs_result.stdout.strip():
    raise RuntimeError('Failed to list APIM subscriptions via az rest.')

subs_payload = json.loads(subs_result.stdout)
subs = subs_payload.get('value', [])
if not subs:
    raise RuntimeError('No APIM subscriptions found to retrieve a key from.')

selected_sub = None
for item in subs:
    scope = item.get('properties', {}).get('scope', '')
    state = item.get('properties', {}).get('state', '')
    if scope.endswith('/products/unlimited') and state == 'active':
        selected_sub = item
        break

if not selected_sub:
    for item in subs:
        if item.get('properties', {}).get('state', '') == 'active':
            selected_sub = item
            break

if not selected_sub:
    selected_sub = subs[0]

sub_name = selected_sub.get('name', '')
if not sub_name:
    raise RuntimeError('Unable to resolve APIM subscription name for key retrieval.')

secret_url = (
    f'https://management.azure.com/subscriptions/{subscription_id}/resourceGroups/{RG_NAME}/'
    f'providers/Microsoft.ApiManagement/service/{apim_name}/subscriptions/{sub_name}/listSecrets?api-version=2022-08-01'
)
secret_result = subprocess.run(
    f'az rest --method post --url "{secret_url}" --query primaryKey -o tsv',
    shell=True,
    capture_output=True,
    text=True
)
if secret_result.returncode != 0 or not secret_result.stdout.strip():
    raise RuntimeError('Failed to retrieve APIM key via az rest listSecrets.')
apim_key = secret_result.stdout.strip()

print('Retrieved Azure OpenAI key and APIM key.')

## Write env/.env

In [ ]:
openai_endpoint = outputs['openAiEndpoint']['value']
apim_endpoint = outputs['apimGatewayUrl']['value']
apim_path = '/openai/chat/completions'
apim_secondary_path = '/openai51/chat/completions'
primary_deployment = outputs['primaryDeploymentName']['value']
secondary_deployment = outputs['secondaryDeploymentName']['value']

subscription_result = subprocess.run('az account show --query id -o tsv', shell=True, capture_output=True, text=True)
subscription_id = subscription_result.stdout.strip() if subscription_result.returncode == 0 else ''

generated_ts = datetime.now(timezone.utc).isoformat()
env_content = f'''# evalgw02 Environment Variables
# Auto-generated by notebooks/01_deploy_infra.ipynb
# Generated UTC: {generated_ts}

AZURE_OPENAI_ENDPOINT={openai_endpoint}
AZURE_OPENAI_API_KEY={openai_key}
AZURE_OPENAI_DEPLOYMENT={primary_deployment}
AZURE_OPENAI_SECONDARY_DEPLOYMENT={secondary_deployment}
AZURE_OPENAI_API_VERSION=2024-10-21

APIM_ENDPOINT={apim_endpoint}
APIM_CHAT_COMPLETIONS_PATH={apim_path}
APIM_SECONDARY_CHAT_COMPLETIONS_PATH={apim_secondary_path}
APIM_API_KEY={apim_key}

AZURE_SUBSCRIPTION_ID={subscription_id}
AZURE_RESOURCE_GROUP={RG_NAME}
AZURE_OPENAI_ACCOUNT_NAME={openai_name}
AZURE_APIM_NAME={apim_name}

EVAL_DATASET_PATH=../data/eval-prompts.jsonl
EVAL_OUTPUT_PATH=../outputs/eval-results.json
'''

env_path = Path('../env/.env')
env_path.write_text(env_content, encoding='utf-8')
print(f'Wrote {env_path}')

## Summary

Deployment completed for one UK South resource group and env/.env has been generated.

Next: Run 02_configure.ipynb